# Assignment 12: Linear Regression - scikit-learn, PyTorch and From Scratch

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

candidates = [
    "50_Startups.csv",
    os.path.join("Learning_Codes", "Assignment_05", "50_Startups.csv"),
    r"c:\Users\debasish.jagaty_jade\ai-master-journey\Learning_Codes\Assignment_05\50_Startups.csv",
]
DATA_PATH = next((p for p in candidates if os.path.exists(p)), candidates[-1])

In [2]:
def load_and_preprocess(path):
    df = pd.read_csv(path)
    df = pd.get_dummies(df, columns=["State"], drop_first=True)
    feature_cols = [c for c in df.columns if c != "Profit"]
    X = df[feature_cols].to_numpy(dtype=float)
    y = df["Profit"].to_numpy(dtype=float)
    return X, y, feature_cols


def standardize_fit(X):
    mean = X.mean(axis=0)
    std = X.std(axis=0)
    std = np.where(std == 0, 1.0, std)
    return mean, std


def standardize_apply(X, mean, std):
    return (X - mean) / std


def gradient_descent(X, y, lr=0.1, epochs=3000):
    if X.shape[0] != y.shape[0]:
        raise ValueError("X and y must have the same number of rows")
    n, m = X.shape
    weights = np.zeros(m)
    bias = 0.0
    for _ in range(epochs):
        preds = X @ weights + bias
        error = preds - y
        grad_w = (2.0 / n) * (X.T @ error)
        grad_b = (2.0 / n) * error.sum()
        weights -= lr * grad_w
        bias -= lr * grad_b
    return weights, bias


def train_torch_linear(X_train, y_train, lr=0.1, epochs=3000, seed=0):
    torch.manual_seed(seed)
    X_t = torch.tensor(X_train, dtype=torch.float32)
    y_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
    model = nn.Linear(X_t.shape[1], 1)
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    for _ in range(epochs):
        optimizer.zero_grad()
        pred = model(X_t)
        loss = loss_fn(pred, y_t)
        loss.backward()
        optimizer.step()
    weight = model.weight.detach().numpy().flatten()
    bias = model.bias.detach().numpy()[0]
    return weight, bias

In [3]:
# sample answer: load the dataset, split, and standardize using train statistics only
X, y, feature_cols = load_and_preprocess(DATA_PATH)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
mean, std = standardize_fit(X_train)
X_train_scaled = standardize_apply(X_train, mean, std)
X_test_scaled = standardize_apply(X_test, mean, std)
print(feature_cols)
print(X_train_scaled.shape, X_test_scaled.shape)

['R&D Spend', 'Administration', 'Marketing Spend', 'State_Florida', 'State_New York']
(40, 5) (10, 5)


In [4]:
# scikit-learn linear regression
sk_model = LinearRegression()
sk_model.fit(X_train_scaled, y_train)
sk_coef = sk_model.coef_
sk_intercept = sk_model.intercept_
sk_test_score = sk_model.score(X_test_scaled, y_test)
print("sklearn coefficients:", sk_coef)
print("sklearn intercept:", sk_intercept)
print("sklearn test r2:", sk_test_score)

sklearn coefficients: [ 3.81022693e+04 -1.86475430e+03  3.38617581e+03  4.47775725e+02
  3.27289103e+00]
sklearn intercept: 115651.72050000001
sklearn test r2: 0.8987266414319839


In [5]:
# pytorch linear regression, trained with plain batch gradient descent (SGD, full batch)
torch_coef, torch_intercept = train_torch_linear(X_train_scaled, y_train, lr=0.1, epochs=3000)
print("pytorch coefficients:", torch_coef)
print("pytorch intercept:", torch_intercept)

pytorch coefficients: [ 3.8102250e+04 -1.8647502e+03  3.3861907e+03  4.4777444e+02
  3.2728920e+00]
pytorch intercept: 115651.7


In [6]:
# linear regression from scratch, own gradient descent loop
scratch_coef, scratch_intercept = gradient_descent(X_train_scaled, y_train, lr=0.1, epochs=3000)
print("scratch coefficients:", scratch_coef)
print("scratch intercept:", scratch_intercept)

scratch coefficients: [ 3.81022693e+04 -1.86475430e+03  3.38617581e+03  4.47775725e+02
  3.27289103e+00]
scratch intercept: 115651.72049999997


In [7]:
# comparison of the three approaches
comparison = pd.DataFrame({
    "feature": feature_cols,
    "sklearn": sk_coef,
    "pytorch": torch_coef,
    "scratch": scratch_coef,
})
print(comparison)
print(f"intercepts - sklearn: {sk_intercept:.4f}, pytorch: {torch_intercept:.4f}, scratch: {scratch_intercept:.4f}")

           feature       sklearn       pytorch       scratch
0        R&D Spend  38102.269270  38102.250000  38102.269270
1   Administration  -1864.754300  -1864.750244  -1864.754300
2  Marketing Spend   3386.175807   3386.190674   3386.175807
3    State_Florida    447.775725    447.774445    447.775725
4   State_New York      3.272891      3.272892      3.272891
intercepts - sklearn: 115651.7205, pytorch: 115651.7031, scratch: 115651.7205


In [8]:
# edge and negative case checks, kept separate from the analysis above

# sanity check against a synthetic dataset with a known exact solution
rng = np.random.default_rng(0)
X_syn = rng.normal(size=(200, 2))
true_w = np.array([2.0, -3.0])
true_b = 5.0
y_syn = X_syn @ true_w + true_b

sk_syn = LinearRegression().fit(X_syn, y_syn)
assert np.allclose(sk_syn.coef_, true_w, atol=1e-6)
assert abs(sk_syn.intercept_ - true_b) < 1e-6

scratch_w, scratch_b = gradient_descent(X_syn, y_syn, lr=0.1, epochs=2000)
assert np.allclose(scratch_w, true_w, atol=1e-2)
assert abs(scratch_b - true_b) < 1e-2

torch_w, torch_b = train_torch_linear(X_syn, y_syn, lr=0.1, epochs=2000)
assert np.allclose(torch_w, true_w, atol=1e-1)
assert abs(torch_b - true_b) < 1e-1

# edge cases for the scratch implementation
zero_epoch_w, zero_epoch_b = gradient_descent(X_syn, y_syn, lr=0.1, epochs=0)
assert np.all(zero_epoch_w == 0) and zero_epoch_b == 0

zero_lr_w, zero_lr_b = gradient_descent(X_syn, y_syn, lr=0.0, epochs=100)
assert np.all(zero_lr_w == 0) and zero_lr_b == 0

try:
    gradient_descent(X_syn, y_syn[:-1], lr=0.1, epochs=10)
    assert False, "expected ValueError for mismatched shapes"
except ValueError:
    pass

# negative case for pytorch: mismatched input size raises a runtime error
try:
    bad_model = nn.Linear(2, 1)
    bad_model(torch.tensor(X_syn[:, :1], dtype=torch.float32))
    assert False, "expected RuntimeError for a mismatched input size"
except RuntimeError:
    pass

# negative case for the data loader
try:
    load_and_preprocess("does_not_exist.csv")
    assert False, "expected FileNotFoundError for a missing file"
except FileNotFoundError:
    pass

# real dataset checks: preprocessing and agreement between the three implementations
assert set(feature_cols) == {"R&D Spend", "Administration", "Marketing Spend", "State_Florida", "State_New York"}
assert not np.isnan(X).any()
assert abs(sk_intercept - y_train.mean()) < 1e-6
assert np.allclose(scratch_coef, sk_coef, rtol=0.1, atol=50)
assert abs(scratch_intercept - sk_intercept) < 50
assert np.allclose(torch_coef, sk_coef, rtol=0.2, atol=200)
assert abs(torch_intercept - sk_intercept) < 200

print("all tests passed")

all tests passed
